# 第4章 股票、基金与市场交易

> **核心问题**：拥有一家企业的一小部分意味着什么？屏幕上的价格如何通过买卖形成？基金为什么不是“另一种股票”？

- 金融线：股权、股息、公司融资、指数、基金/ETF、一级与二级市场、订单和交易成本。
- 数学线：份额、市值加权、收益分解、加权平均和价差。
- Python线：DataFrame、排序、分组、函数、简化订单簿与参数实验。

## AI学习状态

当前进度：第4章开始  
已掌握：现金流、收益率、债权与利率风险  
仍然薄弱：待填写  
下一步：始终区分“企业”“股票”“基金份额”和“市场价格”。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 4.1 股票代表剩余所有权

公司可以通过借债或发行股票获得资金。债权人通常按合同优先获得利息和本金；普通股股东拥有剩余索取权和相应风险，可能获得股息和价格上涨，也可能承担企业价值下降乃至归零。

**一级市场**中发行者获得融资；股票上市后的**二级市场**交易通常发生在投资者之间，企业并不会从每笔二级市场成交中直接获得资金。

### 思考

买入一家上市公司的股票后，为什么不能说“公司欠我一笔固定本金”？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

In [ ]:
company = {
    "company_value": 50_000_000,
    "shares_outstanding": 10_000_000,
    "your_shares": 200,
}
price_per_share = company["company_value"] / company["shares_outstanding"]
ownership = company["your_shares"] / company["shares_outstanding"]

print({"每股价格（简化）": price_per_share,
       "持仓市值": price_per_share * company["your_shares"],
       "持股比例": f"{ownership:.6%}"})

**量化编程警告：价格×股数是市值定义，不是企业“真实价值”的证明**。市场价格会变化，流通股与总股本也需区分，现实估值还涉及债务、现金和未来经营。

## 4.2 股票总回报不只来自价格

若买入价为 $P_0$，期末价为 $P_1$，期间收到每股股息 $D$：

$$R=\frac{P_1-P_0+D}{P_0}$$

只看价格会漏掉股息；真实数据还要处理税费、再投资和公司行动。

In [ ]:
p0, p1, dividend = 20.0, 21.0, 0.6
price_return = p1 / p0 - 1
total_return = (p1 - p0 + dividend) / p0
print({"价格收益率": f"{price_return:.2%}", "含股息总回报": f"{total_return:.2%}"})

## 4.3 指数：把一篮子证券压缩为一个数

指数不是可直接持有的资产，而是按规则汇总一组证券表现的指标。常见规则包括等权、市值加权和价格加权。不同规则回答不同问题。

In [ ]:
stocks = pd.DataFrame({
    "股票": ["甲", "乙", "丙"],
    "期初价格": [10.0, 20.0, 50.0],
    "期末价格": [11.0, 18.0, 52.0],
    "流通股数": [1_000_000, 5_000_000, 500_000],
})
stocks["个股收益率"] = stocks["期末价格"] / stocks["期初价格"] - 1
stocks["期初市值"] = stocks["期初价格"] * stocks["流通股数"]
stocks["市值权重"] = stocks["期初市值"] / stocks["期初市值"].sum()
display(stocks.style.format({"个股收益率": "{:.1%}", "市值权重": "{:.1%}", "期初市值": "{:,.0f}"}))

equal_weight_return = stocks["个股收益率"].mean()
cap_weight_return = np.sum(stocks["个股收益率"] * stocks["市值权重"])
print({"等权指数收益": f"{equal_weight_return:.2%}", "市值加权指数收益": f"{cap_weight_return:.2%}"})

### 观察问题

为什么两个指数收益不同？乙的负收益对哪一种指数影响更大？如果股数数据使用了期末信息，会产生什么问题？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->

## 4.4 基金与ETF

基金汇集投资者资金并持有一组资产。每份基金代表组合的一部分。ETF通常可在交易所买卖；其交易价格可能短暂偏离每份基金资产净值。

基金是否分散、风险多高，取决于底层资产和策略。单一行业ETF仍可能高度集中。上交所投教将ETF解释为可在交易所交易、追踪特定指数并具有申购赎回机制的基金。

In [ ]:
holdings = pd.DataFrame({"资产": ["股票甲", "股票乙", "债券丁", "现金"],
                         "市值": [4_000_000, 3_000_000, 2_500_000, 500_000]})
liabilities = 100_000
fund_shares = 2_000_000
nav = (holdings["市值"].sum() - liabilities) / fund_shares
market_price = 5.02
premium = market_price / nav - 1
print({"每份净值NAV": round(nav, 4), "市场价格": market_price, "溢价率": f"{premium:.2%}"})

**Python提示：先计算组合总资产，再减负债，最后除以份额**。每一步都可以打印并检查，避免把“每份价格”和“基金总市值”混为一谈。

## 4.5 订单簿：价格来自愿意交易的买卖双方

限价买单给出“最多愿意付多少”，限价卖单给出“至少愿意收多少”。最高买价称最佳买价，最低卖价称最佳卖价；两者之差是买卖价差。

In [ ]:
orders = pd.DataFrame([
    {"方向": "买", "价格": 9.98, "数量": 500},
    {"方向": "买", "价格": 9.96, "数量": 800},
    {"方向": "买", "价格": 9.95, "数量": 600},
    {"方向": "卖", "价格": 10.02, "数量": 300},
    {"方向": "卖", "价格": 10.04, "数量": 700},
    {"方向": "卖", "价格": 10.06, "数量": 900},
])
bids = orders[orders["方向"] == "买"].sort_values("价格", ascending=False)
asks = orders[orders["方向"] == "卖"].sort_values("价格")
best_bid, best_ask = bids.iloc[0]["价格"], asks.iloc[0]["价格"]
print({"最佳买价": best_bid, "最佳卖价": best_ask,
       "价差": round(best_ask - best_bid, 4), "中间价": (best_bid + best_ask) / 2})
display(pd.concat([bids, asks]))

### 市价买单模拟

市价买单会从最低卖价开始逐档成交。订单越大，可能吃掉更多价位，平均成交价上升。这是最简化的市场冲击直觉。

In [ ]:
def execute_market_buy(asks, quantity):
    remaining = quantity
    trades = []
    for row in asks.itertuples():
        filled = min(remaining, row.数量)
        if filled > 0:
            trades.append({"价格": row.价格, "成交数量": filled})
            remaining -= filled
        if remaining == 0:
            break
    trade_table = pd.DataFrame(trades)
    if remaining > 0:
        raise ValueError("卖盘数量不足，无法全部成交")
    average = np.average(trade_table["价格"], weights=trade_table["成交数量"])
    return trade_table, average


trades, average_price = execute_market_buy(asks, 800)
display(trades)
print({"平均成交价": round(average_price, 4),
       "相对最佳卖价滑点": f"{average_price / best_ask - 1:.3%}"})

**量化编程警告：真实撮合远比这里复杂**。订单会到达、撤销和排队；交易规则、涨跌幅、最小价位、费用和交收制度因市场与产品而异，实际使用前必须查阅交易所最新规则。

In [ ]:
quantities = np.arange(100, asks["数量"].sum() + 1, 100)
averages = [execute_market_buy(asks, int(q))[1] for q in quantities]
plt.step(quantities, averages, where="post")
plt.axhline(best_ask, color="gray", linestyle="--", label="最佳卖价")
plt.xlabel("市价买入数量"); plt.ylabel("平均成交价"); plt.title("订单规模与平均成交价（简化订单簿）")
plt.legend(); plt.show()

## 4.6 交易成本会改变收益

除显式佣金外，价差、滑点和市场冲击也会消耗收益。回测若统一按收盘价无限成交，会夸大可实现性。

In [ ]:
signal_price = 10.00
buy_price = average_price
sell_price = 10.40
commission_rate = 0.0003
quantity = 800

gross_pnl = (sell_price - buy_price) * quantity
commissions = (buy_price + sell_price) * quantity * commission_rate
net_pnl = gross_pnl - commissions
print({"信号价": signal_price, "实际买入均价": round(buy_price, 4),
       "毛利润": round(gross_pnl, 2), "佣金": round(commissions, 2), "净利润": round(net_pnl, 2)})

## 4.7 编程练习：实现市值加权组合收益

要求检查权重与收益长度相同、权重和接近1，并返回加权收益。

In [ ]:
def weighted_return(returns, weights):
    # TODO：转换数组、检查形状和权重，再计算点积
    return None

In [ ]:
answer = weighted_return([0.10, -0.10, 0.04], [0.2, 0.6, 0.2])
if answer is None:
    print("练习尚未完成。")
else:
    print("基础测试通过：", np.isclose(answer, -0.032))

### 我的解释

为什么“买入指数基金”不等于直接买入指数？ETF市场价格偏离NAV、基金费用和跟踪误差分别可能带来什么差异？

<!-- 在这里填写；完成前AI不要代答 -->

### AI批改区

<!-- 检查概念边界、权重时点、费用和订单可成交性。 -->

## 本章总结与小项目

构造5只虚拟股票和两个指数，比较等权与市值加权；再用订单簿模拟不同规模交易，报告价差、滑点和费用。解释为什么“指数上涨”不代表每只成分股都上涨。

**参考**：上海证券交易所投资者教育（证券基础、ETF和债券专题）；Investor.gov Stocks与Mutual Funds。